In [1]:
import os
os.environ["GOOGLE_API_KEY"] = "AQ.Ab8RN6LYxobf-OOxdVkEnqdgnS2iwQEWVVFHHrvfvFoy3muZMQ"

In [2]:
!uv pip install -qU langchain langchain-google-genai langchain-chroma langchain-community langgraph pypdf tiktoken

In [13]:
import ast
import operator
import os

from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_google_genai import (
    ChatGoogleGenerativeAI,
    GoogleGenerativeAIEmbeddings
)
from langchain_chroma import Chroma
from langchain_core.tools import tool
from langchain_core.documents import Document
from langgraph.prebuilt import create_react_agent
from langgraph.checkpoint.memory import MemorySaver


# ============================================================
# 1. DATASET
# ============================================================

docs = [
    Document(
        page_content=(
            "Artificial intelligence allows computers to perform "
            "tasks that normally require human intelligence."
        ),
        metadata={
            "source": "dataset",
            "topic": "AI",
            "id": 1
        }
    ),

    Document(
        page_content=(
            "Machine learning is a branch of artificial "
            "intelligence where computers learn patterns from data."
        ),
        metadata={
            "source": "dataset",
            "topic": "Machine Learning",
            "id": 2
        }
    ),

    Document(
        page_content=(
            "Retrieval Augmented Generation allows a language "
            "model to retrieve external information before "
            "generating an answer."
        ),
        metadata={
            "source": "dataset",
            "topic": "RAG",
            "id": 3
        }
    ),

    Document(
        page_content=(
            "Vector databases store vector representations "
            "of information and allow similarity-based searching."
        ),
        metadata={
            "source": "dataset",
            "topic": "Vector Database",
            "id": 4
        }
    )
]


# ============================================================
# 2. CHUNKING
# ============================================================

splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

chunks = splitter.split_documents(docs)

print("Number of documents:", len(docs))
print("Number of chunks:", len(chunks))


# ============================================================
# 3. GOOGLE EMBEDDINGS
# ============================================================

embeddings = GoogleGenerativeAIEmbeddings(
    model="gemini-embedding-2-preview"
)


# ============================================================
# 4. CHROMA VECTOR DATABASE
# ============================================================

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings
)

print("Chroma vector database created!")


# ============================================================
# 5. RETRIEVER
# ============================================================

retriever = vectorstore.as_retriever(
    search_kwargs={"k": 2}
)


# ============================================================
# 6. DATASET SEARCH TOOL
# ============================================================

@tool
def search_dataset(query: str) -> str:
    """
    Search the dataset for factual information,
    context, and relevant information.
    """

    results = retriever.invoke(query)

    if not results:
        return "No relevant information found."

    return "\n\n".join(
        doc.page_content
        for doc in results
    )


# ============================================================
# 7. SAFE CALCULATOR
# ============================================================

def safe_eval(expr: str):
    """
    A restricted mathematical evaluator.
    """

    allowed_operators = {
        ast.Add: operator.add,
        ast.Sub: operator.sub,
        ast.Mult: operator.mul,
        ast.Div: operator.truediv,
        ast.USub: operator.neg,
        ast.Pow: operator.pow
    }

    def _eval(node):

        if isinstance(node, ast.Constant):
            return node.value

        elif isinstance(node, ast.BinOp):

            if type(node.op) not in allowed_operators:
                raise TypeError("Unsupported math operation")

            return allowed_operators[type(node.op)](
                _eval(node.left),
                _eval(node.right)
            )

        elif isinstance(node, ast.UnaryOp):

            if type(node.op) not in allowed_operators:
                raise TypeError("Unsupported math operation")

            return allowed_operators[type(node.op)](
                _eval(node.operand)
            )

        else:
            raise TypeError("Unsupported expression")

    tree = ast.parse(expr, mode="eval").body

    return _eval(tree)


# ============================================================
# 8. CALCULATOR TOOL
# ============================================================

@tool
def calculator(expression: str) -> str:
    """
    Evaluate a mathematical expression.

    Example:
    25000 * 0.15
    """

    try:

        result = safe_eval(expression)

        return f"Calculation Result: {result}"

    except Exception as e:

        return f"Error evaluating: {e}"


# ============================================================
# 9. INITIALIZE GEMINI AGENT WITH MEMORY
# ============================================================

print("Initializing Gemini Agent...")

model = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash-lite",
    temperature=0
)

memory = MemorySaver()


# ============================================================
# 10. CREATE THE AGENT
# ============================================================

agent = create_react_agent(
    model,
    tools=[
        search_dataset,
        calculator
    ],
    checkpointer=memory
)


# ============================================================
# 11. CREATE A CONVERSATION THREAD
# ============================================================

config = {
    "configurable": {
        "thread_id": "jupyter_session_1"
    }
}


# ============================================================
# 12. ASK THE AGENT A QUESTION
# ============================================================

user_query = """
Search the dataset and explain what
Retrieval Augmented Generation (RAG) is.
"""

print(f"\nUser: {user_query}")

print("Agent is thinking and using tools...\n")


# ============================================================
# 13. RUN THE AGENT
# ============================================================

result = agent.invoke(
    {
        "messages": [
            ("user", user_query)
        ]
    },
    config=config
)


# ============================================================
# 14. DISPLAY FINAL ANSWER
# ============================================================

print("--- Final Answer ---")

print(
    result["messages"][-1].content
)

Number of documents: 4
Number of chunks: 4
Chroma vector database created!
Initializing Gemini Agent...

User: 
Search the dataset and explain what
Retrieval Augmented Generation (RAG) is.

Agent is thinking and using tools...



C:\Users\AB\AppData\Local\Temp\ipykernel_11788\4224269897.py:232: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(
E:\AI-ML\.venv310\lib\site-packages\langchain_google_genai\chat_models.py:3237: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
E:\AI-ML\.venv310\lib\site-packages\langchain_google_genai\chat_models.py:3237: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


--- Final Answer ---
[{'type': 'text', 'text': 'Based on the dataset, **Retrieval-Augmented Generation (RAG)** is a technique that allows a language model to retrieve external information before generating an answer. \n\nIt typically works in conjunction with **vector databases**, which store vector representations of information and enable similarity-based searching to quickly find relevant context and facts to feed into the model.', 'extras': {'signature': 'El4KXAERTTIPJAS0woVZ2XnoV9eVjsbylNK+3AmaHKqah5y9HR5PllRKhi4CykU6rmb7Piu2xaCxyipg2ymudEcP7ctFjF2w6POkhqTSvosaZBmJzbDGLPhqqSB7l0RK'}}]


In [14]:
# 5. Run the Agent (Streaming Version)
user_query = "Search the PDF for the total revenue or any major financial number. Once you find it, calculate what a 15% increase on that amount would be."
print(f"\nUser: {user_query}")
print("--- Agent is thinking (Live Stream) ---\n")

# Use .stream() instead of .invoke()
for chunk in agent.stream({"messages": [("user", user_query)]}, config=config):
    
    # LangGraph chunks are dictionaries keyed by the node name that just ran
    for node_name, state_update in chunk.items():
        print(f"👉 [Node: {node_name.upper()}]")
        
        # Grab the most recent message generated by this step
        latest_message = state_update["messages"][-1]
        
        # 1. Did the agent decide to call a tool?
        if hasattr(latest_message, "tool_calls") and latest_message.tool_calls:
            for tool_call in latest_message.tool_calls:
                print(f"   🔧 Tool Requested: {tool_call['name']}")
                print(f"   📥 Inputs: {tool_call['args']}")
        
        # 2. Did a tool just finish running and return a result?
        elif getattr(latest_message, "type", "") == "tool":
            print(f"   ✅ Tool Result: {latest_message.content[:200]}... [truncated]")
            
        # 3. Did the agent output standard text?
        elif latest_message.content:
            print(f"   💬 Agent: {latest_message.content}\n")
        
        print("-" * 40)


User: Search the PDF for the total revenue or any major financial number. Once you find it, calculate what a 15% increase on that amount would be.
--- Agent is thinking (Live Stream) ---



E:\AI-ML\.venv310\lib\site-packages\langchain_google_genai\chat_models.py:3237: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


👉 [Node: AGENT]
   🔧 Tool Requested: search_dataset
   📥 Inputs: {'query': 'revenue financial total'}
----------------------------------------
👉 [Node: TOOLS]
   ✅ Tool Result: The future is built through ordinary moments. A few minutes of reading, one programming
exercise, one conversation, one workout, or one attempt at something difficult may seem
insignificant by itself.... [truncated]
----------------------------------------


E:\AI-ML\.venv310\lib\site-packages\langchain_google_genai\chat_models.py:3237: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


👉 [Node: AGENT]
   🔧 Tool Requested: search_dataset
   📥 Inputs: {'query': 'dollar number financial revenue profit sales'}
----------------------------------------
👉 [Node: TOOLS]
   ✅ Tool Result: Over time, small steps become progress, progress becomes skill, skill becomes confidence, and
confidence can eventually become achievement. The journey may be slow, but every meaningful
journey begins... [truncated]
----------------------------------------


E:\AI-ML\.venv310\lib\site-packages\langchain_google_genai\chat_models.py:3237: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


👉 [Node: AGENT]
   💬 Agent: [{'type': 'text', 'text': 'I searched the dataset for any financial information, revenue figures, or major financial numbers, but the document appears to be a motivational text about personal growth, small steps, and habits rather than a corporate or financial report. Consequently, there is no revenue figure available to calculate a 15% increase on.', 'extras': {'signature': 'El4KXAERTTIPQlQ27Bw+s3qrnDI/abOM4y2l6W8UTgSvZ4Mbx5JgGrqLb7vjy8Y7j5Rk/rgELPrRX42nJ0EkBpo4i1f5z1OPOwNX4AwyjhNxBjWGCloN4inpxyBBCEqa'}}]

----------------------------------------


In [15]:
# The agent should remember what "that answer" refers to without running tools
follow_up = "Can you format that answer into a concise 2-bullet summary?"

for chunk in agent.stream({"messages": [("user", follow_up)]}, config=config):
    for node_name, state_update in chunk.items():
        latest_message = state_update["messages"][-1]
        if latest_message.content:
            print(latest_message.content)

E:\AI-ML\.venv310\lib\site-packages\langchain_google_genai\chat_models.py:3237: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


[{'type': 'text', 'text': '* **No Financial Data Found:** The dataset contains motivational text about personal growth rather than financial reports or revenue numbers.\n* **Calculation Unavailable:** Because no financial figures are present, it is not possible to calculate a 15% increase.', 'extras': {'signature': 'El4KXAERTTIPAgqaCBfTSi4x7Ks9PIMu18LoigCXS/gVctwaRf1T0BuSNa5SHzoNnEvu4PcpOpuYU2Yputrqry2qgbcnTuOO8+rOUstAFK4fmApKB5z/lmmTm3NjJ8l3'}}]


In [16]:
# Change your vectorstore setup to save locally:
vectorstore = Chroma.from_documents(
    documents=chunks, 
    embedding=embeddings,
    persist_directory="./chroma_db"  # Saves database to disk
)

In [17]:
# Interrupt the execution loop before running tools
agent = create_react_agent(
    model, 
    tools=[search_pdf, calculator], 
    checkpointer=memory,
    interrupt_before=["tools"]  # Pauses for human confirmation
)

C:\Users\AB\AppData\Local\Temp\ipykernel_11788\2467535410.py:2: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(


In [18]:
import datetime

@tool
def get_current_time() -> str:
    """Returns the current date and time."""
    return datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")

# Add get_current_time to your tools list when creating the agent!